In [1]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'GEMINI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [20]:
from google import genai
client=genai.Client()
from google.genai import types
import json
import time
import traceback
from typing import List, Callable, Dict, Any
from dataclasses import dataclass, field



@dataclass
class Prompt:
    messages: List[Dict] = field(default_factory=list)
    tools: List[Dict] = field(default_factory=list)
    metadata: dict = field(default_factory=dict)  # Fixing mutable default issue



def generate_response(prompt: Prompt) -> str:
    """Call Gemini LLM to get response (handles message & tool translation)"""

    messages = prompt.messages
    tools = prompt.tools

    system_instruction = agent_rules[0]["content"] if 'agent_rules' in globals() and agent_rules else None

    # 1. Convert OpenAI-style messages to Gemini's contents format
    gemini_contents = []
    for msg in messages:
        # Handle dict or object message structures
        role = msg.get("role") if isinstance(msg, dict) else getattr(msg, "role", "user")
        content = msg.get("content") if isinstance(msg, dict) else getattr(msg, "content", "")

        # Gemini uses 'model' instead of OpenAI's 'assistant'
        if role == "assistant":
            role = "model"
        elif role == "system":
            # System instructions are handled in config, skip them here if present in messages
            continue

        gemini_contents.append({
            "role": role,
            "parts": [{"text": str(content)}]
        })

    config_params = {
        "system_instruction": system_instruction,
        "max_output_tokens": 1024,
    }

    # 2. Convert OpenAI-style tools to Gemini's native Tool format
    if tools:
        gemini_tools = []
        for t in tools:
            if isinstance(t, dict) and "function" in t:
                func = t["function"]
                gemini_tools.append(
                    types.Tool(
                        function_declarations=[
                            types.FunctionDeclaration(
                                name=func.get("name"),
                                description=func.get("description"),
                                parameters=func.get("parameters")
                            )
                        ]
                    )
                )
            else:
                gemini_tools.append(t)

        config_params["tools"] = gemini_tools

    # 3. Call Gemini model with translated contents and config
    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=gemini_contents,
        config=types.GenerateContentConfig(**config_params)
    )

    if response.function_calls:
        tool_call = response.function_calls[0]
        result = {
            "tool": tool_call.name,
            "args": tool_call.args,
        }
        return json.dumps(result)

    return response.text

@dataclass(frozen=True)
class Goal:
    priority: int
    name: str
    description: str



class Action:
  def __init__(self,
               name:str,
               function:callable,
               description:str,
               parameters:Dict,
               terminal:bool=False):
    self.name=name
    self.function=function
    self.description=description
    self.parameters=parameters
    self.terminal=terminal

  def execute(self,**args)->Any:
    #execute function call
    return self.function(**args)

class ActionRegistry:
  def __init__(self):
    self.actions={}

  def register(self,action:Action):
    self.actions[action.name]=action

  def get_action(self, name: str) -> [Action, None]:
    return self.actions.get(name,None)

  def get_actions(self)->List[Action]:
    return list(self.actions.values())


In [21]:

class Memory:
  def __init__(self):
    self.items=[]

  def add_memory(self,memory:dict):
    self.items.append(memory)

  def get_memories(self,limit:int=None)->List[Dict]:
    if limit is None:
        return self.items
    limit=min(limit,len(self.items))
    return self.items[-limit:]

class Environment:
  def execute_action(self,action:Action,args:dict)->dict:
    try:
      result=action.execute(**args)
      return self.format_result(result)
    except Exception as e:
      return {
          "tool_executed":False,
          "error":str(e),
          "traceback":traceback.format_exc()
      }
  def format_result(self,result:Any)->dict:
    return {
        "tool_executed":True,
        "result":result,
        "timestamp":time.strftime("%Y-%m_%dT%H:%M:%S%z")
    }

In [22]:
class AgentLanguage:
    def __init__(self):
        pass

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:
        raise NotImplementedError("Subclasses must implement this method")


    def parse_response(self, response: str) -> dict:
        raise NotImplementedError("Subclasses must implement this method")



class AgentFunctionCallingActionLanguage(AgentLanguage):

    def __init__(self):
        super().__init__()

    def format_goals(self, goals: List[Goal]) -> List:
        # Map all goals to a single string that concatenates their description
        # and combine into a single message of type system
        sep = "\n-------------------\n"
        goal_instructions = "\n\n".join([f"{goal.name}:{sep}{goal.description}{sep}" for goal in goals])
        return [
            {"role": "system", "content": goal_instructions}
        ]

    def format_memory(self, memory: Memory) -> List:
        """Generate response from language model"""
        # Map all environment results to a role:user messages
        # Map all assistant messages to a role:assistant messages
        # Map all user messages to a role:user messages
        items = memory.get_memories()
        mapped_items = []
        for item in items:

            content = item.get("content", None)
            if not content:
                content = json.dumps(item, indent=4)

            if item["type"] == "assistant":
                mapped_items.append({"role": "assistant", "content": content})
            elif item["type"] == "environment":
                mapped_items.append({"role": "assistant", "content": content})
            else:
                mapped_items.append({"role": "user", "content": content})

        return mapped_items

    def format_actions(self, actions: List[Action]) -> [List,List]:
        """Generate response from language model"""

        tools = [
            {
                "type": "function",
                "function": {
                    "name": action.name,
                    # Include up to 1024 characters of the description
                    "description": action.description[:1024],
                    "parameters": action.parameters,
                },
            } for action in actions
        ]

        return tools

    def construct_prompt(self,
                         actions: List[Action],
                         environment: Environment,
                         goals: List[Goal],
                         memory: Memory) -> Prompt:

        prompt = []
        prompt += self.format_goals(goals)
        prompt += self.format_memory(memory)

        tools = self.format_actions(actions)

        return Prompt(messages=prompt, tools=tools)

    def adapt_prompt_after_parsing_error(self,
                                         prompt: Prompt,
                                         response: str,
                                         traceback: str,
                                         error: Any,
                                         retries_left: int) -> Prompt:

        return prompt

    def parse_response(self, response: str) -> dict:
        """Parse LLM response into structured format by extracting the ```json block"""

        try:
            return json.loads(response)

        except Exception as e:
            return {
                "tool": "terminate",
                "args": {"message":response}
            }

class Agent:
    def __init__(self,
                 goals: List[Goal],
                 agent_language: AgentLanguage,
                 action_registry: ActionRegistry,
                 generate_response: Callable[[Prompt], str],
                 environment: Environment):
        """
        Initialize an agent with its core GAME components
        """
        self.goals = goals
        self.generate_response = generate_response
        self.agent_language = agent_language
        self.actions = action_registry
        self.environment = environment

    def construct_prompt(self, goals: List[Goal], memory: Memory, actions: ActionRegistry) -> Prompt:
        """Build prompt with memory context"""
        return self.agent_language.construct_prompt(
            actions=actions.get_actions(),
            environment=self.environment,
            goals=goals,
            memory=memory
        )

    def get_action(self, response):
        invocation = self.agent_language.parse_response(response)
        action = self.actions.get_action(invocation["tool"])
        return action, invocation

    def should_terminate(self, response: str) -> bool:
        action_def, _ = self.get_action(response)
        return action_def.terminal

    def set_current_task(self, memory: Memory, task: str):
        memory.add_memory({"type": "user", "content": task})

    def update_memory(self, memory: Memory, response: str, result: dict):
        """
        Update memory with the agent's decision and the environment's response.
        """
        new_memories = [
            {"type": "assistant", "content": response},
            {"type": "user", "content": json.dumps(result)}
        ]
        for m in new_memories:
            memory.add_memory(m)

    def prompt_llm_for_action(self, full_prompt: Prompt) -> str:
        response = self.generate_response(full_prompt)
        return response

    def run(self, user_input: str, memory=None, max_iterations: int = 50) -> Memory:
        """
        Execute the GAME loop for this agent with a maximum iteration limit.
        """
        memory = memory or Memory()
        self.set_current_task(memory, user_input)

        for _ in range(max_iterations):
            # Construct a prompt that includes the Goals, Actions, and the current Memory
            prompt = self.construct_prompt(self.goals, memory, self.actions)

            print("Agent thinking...")
            # Generate a response from the agent
            response = self.prompt_llm_for_action(prompt)
            print(f"Agent Decision: {response}")

            # Determine which action the agent wants to execute
            action, invocation = self.get_action(response)

            # Execute the action in the environment
            result = self.environment.execute_action(action, invocation["args"])
            print(f"Action Result: {result}")

            # Update the agent's memory with information about what happened
            self.update_memory(memory, response, result)

            # Check if the agent has decided to terminate
            if self.should_terminate(response):
                break

        return memory

In [24]:
def list_files()->list:
  return os.listdir(".")
def read_file(file_name:str)->str:
  with open(file_name,'r') as f:
    return f.read()
def search_in_file(file_name:str,search_term:str)->list:
  results=[]
  with open(file_name,'r') as f:
    for i,line in enumerate(f.readlines()):
      if search_term in line:
        results.append(i+1,line.strip())
  return results
def terminate(message: str) -> str:
        """Terminate the agent loop and provide a summary message."""
        return message

registry=ActionRegistry()

registry.register(Action(
    name="list_files",
    function=list_files,
    description="list all fiels in the current directiory",
    parameters={
        "type":"object",
        "properties":{},
        "required":[]
    },
    terminal=False
))

registry.register(Action(
    name="read_file",
    function=read_file,
    description="Read the contents of a specific file",
    parameters={
        "type": "object",
        "properties": {
            "file_name": {
                "type": "string",
                "description": "Name of the file to read"
            }
        },
        "required": ["file_name"]
    },
    terminal=False
))

registry.register(Action(
    name="search_in_file",
    function=search_in_file,
    description="search for a term in a specific file",
    parameters={
        "type":"object",
        "properties":{
            "file_name":{
                "type":"string",
                "description":"Name of the file to search in"
            },
            "search_term":{
                "type":"string",
                "description":"Term to search for"
            }
        },
        "required":["file_name","search_term"]
    },
    terminal=False
))
registry.register(Action(
        name="terminate",
        function=terminate,
        description="Terminates the conversation. Prints the provided message for the user.",
        parameters={
            "type": "object",
            "properties": {
                "message": {"type": "string"},
            },
            "required": ["message"]
        },
        terminal=True
    ))

file_management_goal=goals = [
        Goal(
            priority=1,
            name="Explore Files",
            description="Explore files in the current directory by listing and reading them"
        ),
        Goal(
            priority=2,
            name="Terminate",
            description="Terminate the session when tasks are complete with a helpful summary"
        )
    ]
agent_language=AgentFunctionCallingActionLanguage()
environment=Environment()

file_explorer_agent = Agent(
        goals=file_management_goal,
        agent_language=agent_language,
        action_registry=registry,
        generate_response=generate_response,
        environment=environment
    )



user_input = input("What would you like me to do? ")

final_memory = file_explorer_agent.run(user_input, max_iterations=10)

# Print the termination message (if any)

for item in final_memory.get_memories():
        print(f"\nMemory: {item['content']}")


What would you like me to do? Tell me what Python files are in this directory and summarize how they fit together.
Agent thinking...
Agent Decision: {"tool": "list_files", "args": {}}
Action Result: {'tool_executed': True, 'result': ['.config', 'a.py', 'sample_data'], 'timestamp': '2026-11_07T20:01:22+0000'}
Agent thinking...
Agent Decision: {"tool": "read_file", "args": {"file_name": "a.py"}}
Action Result: {'tool_executed': True, 'result': 'import re\nfrom sympy import Matrix\n\ndef clean_text(text: str) -> str:\n    return re.sub(r\'[^A-Za-z]\', \'\', text).upper()\n\ndef encrypt_hill(plaintext: str, key: Matrix) -> str:\n    n = key.shape[0]\n    clean = clean_text(plaintext)\n    \n    # Pad if necessary\n    pad = (n - (len(clean) % n)) % n\n    clean += \'X\' * pad\n    \n    ciphertext = []\n    for i in range(0, len(clean), n):\n        # Create column vector\n        p_vec = Matrix([ord(c) - ord(\'A\') for c in clean[i:i+n]])\n        \n        # C = K * P mod 26\n        c_v